In [9]:
#imports
import mne
import glob
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from pathlib import Path

In [10]:
r_dir = '/Users/madhav/PSY197B'

source_dir_eeg   = os.path.join(r_dir, 'data', 'sj04', 'eeg')
source_dir_trial = os.path.join(r_dir, 'data', 'sj04', 'beh')
source_dir_et    = os.path.join(r_dir, 'data', 'sj04', 'eye')

dest_dir      = os.path.join(r_dir, 'EEG_Prepro_Test')
dest_dir_plot = os.path.join(dest_dir, 'Plots', 'Individual')

for d in [dest_dir, dest_dir_plot]:
    os.makedirs(d, exist_ok=True)

subjects = [4]

conditions = [
    {'eeg_label': 'walk_attend',   'trial_label': 'Attend_Walk'},
    {'eeg_label': 'walk_unattend', 'trial_label': 'Unattend_Walk'},
]

# Maps eeg_label → actual Pupil Labs export folder name
et_folder_map = {
    'walk_attend':   'Attend_Walk',
    'walk_unattend': 'Unattend_Walk',
    'sit_attend':    'Attend_Sit',
    'sit_unattend':  'Unattend_Sit',
}

In [11]:
def align_to_eeg_events(df, eeg_event_list, idx_col='trialIdx'):
    """Align a DataFrame to EEG event order by matching on idx_col."""
    missing = set(df[idx_col]) - set(eeg_event_list)
    if missing:
        print(f'    Removing {len(missing)} trials missing from EEG events')
        df = df[df[idx_col].isin(eeg_event_list)].reset_index(drop=True)

    idx_map      = {v: i for i, v in enumerate(df[idx_col])}
    aligned_rows = [idx_map.get(code) for code in eeg_event_list]

    if None in aligned_rows:
        print('    SYNC FAIL — EEG event(s) not found in trialIdx')
        return None

    aligned = df.iloc[aligned_rows].reset_index(drop=True)
    if not np.all(aligned[idx_col].values == eeg_event_list):
        print('    SYNC FAIL — final check mismatch')
        return None

    print('    SYNC SUCCESS')
    return aligned


def parse_annotations(annotations):
    """
    Parse Pupil Labs annotations.csv into a clean triggers DataFrame.
    
    Format:  label = 'Event_N',  timestamp = nanoseconds
    Keeps:   trial triggers (1–200)
    Excludes: status/sync codes e.g. Event_201, Event_222
    """
    annotations['trigger']     = annotations['label'].str.extract(r'Event_(\d+)').astype(float)
    annotations['timestamp_s'] = annotations['timestamp [ns]'] / 1e9

    triggers = annotations.dropna(subset=['trigger']).copy()
    triggers['trigger'] = triggers['trigger'].astype(int)

    # Report excluded non-trial codes for awareness
    excluded = triggers[~triggers['trigger'].between(1, 200)]['label'].unique()
    if len(excluded):
        print(f'    Non-trial event codes excluded (check meanings): {excluded}')

    triggers = triggers[triggers['trigger'].between(1, 200)].copy()
    triggers = triggers.rename(columns={'timestamp_s': 'start_timestamp'}).reset_index(drop=True)
    return triggers


In [12]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
for i_sub in range(len(subjects)):
    sj_num = subjects[i_sub]
    print(f'\nProcessing Subject {sj_num}...')

    for i_cond, cond in enumerate(conditions):
        this_label_eeg   = cond['eeg_label']
        this_label_trial = cond['trial_label']
        print(f'  Processing condition: {this_label_eeg}')

        # Load trial data across 5 blocks
        trial_data_list = []
        for i_block in range(1, 6):
            filename = os.path.join(
                source_dir_trial,
                f'sj{sj_num:02d}_block{i_block}_{this_label_trial}.csv'
            )
            if os.path.exists(filename):
                trial_data_list.append(pd.read_csv(filename))
            else:
                print(f'    Warning: No file found for block {i_block}: {filename}')

        if not trial_data_list:
            print(f'    Error: No trial data found for {this_label_trial}')
            continue

        trial_data = pd.concat(trial_data_list, ignore_index=True)
        print(f'    Loaded {len(trial_data)} trials')

        eeg_file = os.path.join(source_dir_eeg, f'sj{sj_num:02d}_{this_label_eeg}.vhdr')
        if not os.path.exists(eeg_file):
            print(f'    Error: EEG file not found: {eeg_file}')
            continue

        print(f'    Loading EEG from: {eeg_file}')
        raw = mne.io.read_raw_brainvision(eeg_file, preload=True)

        if raw.info['sfreq'] != 250:
            print(f'    Downsampling from {raw.info["sfreq"]:.1f} Hz to 250 Hz')
            raw = raw.resample(250)

        if 'TP9' in raw.ch_names and 'TP10' in raw.ch_names:
            print('    Re-referencing to average of mastoids (TP9, TP10)')
            raw.set_eeg_reference(['TP9', 'TP10'], ch_type='eeg')
        else:
            print('    Warning: TP9 or TP10 not found, skipping re-reference')

        ch_to_remove = [
            ch for ch in raw.ch_names
            if any(x in ch.lower() for x in ['x_dir', 'y_dir', 'z_dir', 'r_x', 'r_y', 'r_z', 'l_x', 'l_y', 'l_z'])
            or ch == '32'
        ]
        if ch_to_remove:
            print(f'    Removing channels: {ch_to_remove}')
            raw.drop_channels(ch_to_remove)

        print('    Filtering: 0.1–30 Hz')
        raw.filter(0.1, 30, fir_design='firwin2')

        eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
        if len(eeg_picks) > 0:
            data_eeg  = raw.get_data(picks=eeg_picks)
            chan_std   = np.std(data_eeg, axis=1)
            z_scores   = (chan_std - np.mean(chan_std)) / (np.std(chan_std) + 1e-12)
            bad_chans  = [raw.ch_names[eeg_picks[i]] for i, z in enumerate(z_scores) if z > 3.5]
            if bad_chans:
                print(f'    Marking bad channels (z>3.5): {bad_chans}')
                raw.info['bads'].extend(bad_chans)
                raw.interpolate_bads(reset_bads=True)

        ica = mne.preprocessing.ICA(n_components=0.99, method='fastica', random_state=97, max_iter='auto')
        ica.fit(raw, picks='eeg')
        raw = ica.apply(raw)

        events, event_id = mne.events_from_annotations(raw)

        print('    Adjusting trigger latencies (+60 samples for triggers <= 200)')
        events_adjusted = events.copy()
        events_adjusted[events_adjusted[:, 2] <= 200, 0] += 60

        tmin, tmax = -0.2, 1.0
        valid_events   = events_adjusted[events_adjusted[:, 2] <= 200]
        valid_event_ids = {str(code): code for code in np.unique(valid_events[:, 2])}

        epochs = mne.Epochs(raw, valid_events, event_id=valid_event_ids,
                            tmin=tmin, tmax=tmax, baseline=None,
                            preload=True, verbose=False)
        epochs.apply_baseline(baseline=(-0.2, 0))
        print(f'    Created {len(epochs)} epochs')

        eeg_event_list       = epochs.events[:, 2]
        trial_data_event_list = trial_data['trialIdx'].values

        print(f'    Checking sync: EEG={len(eeg_event_list)}, Trial={len(trial_data_event_list)}')

        missing_in_eeg = set(trial_data_event_list) - set(eeg_event_list)
        if missing_in_eeg:
            print(f'    Removing {len(missing_in_eeg)} trials with missing EEG triggers')
            trial_data = trial_data[trial_data['trialIdx'].isin(eeg_event_list)].reset_index(drop=True)

        trial_data = align_to_eeg_events(trial_data, eeg_event_list, idx_col='trialIdx')
        if trial_data is None:
            print('    ABORT — EEG/trial sync failed')
            continue

        if len(trial_data) == len(epochs):
            epochs.metadata = trial_data.copy()
            print('    Added trial data as metadata')
        else:
            print(f'    WARNING: metadata length mismatch ({len(trial_data)} vs {len(epochs)})')

        epo_path   = os.path.join(dest_dir, f'sj{sj_num:02d}_{this_label_eeg}_EEG_Prepro1-epo.fif')
        trial_path = os.path.join(dest_dir, f'sj{sj_num:02d}_{this_label_eeg}_trialData.csv')
        epochs.save(epo_path, overwrite=True)
        trial_data.to_csv(trial_path, index=False)
        print(f'    Saved: {epo_path}')
        print(f'    Saved: {trial_path}')

print('\nEEG preprocessing complete!')


Processing Subject 4...
  Processing condition: walk_attend
    Loaded 1000 trials
    Loading EEG from: /Users/madhav/PSY197B/data/sj04/eeg/sj04_walk_attend.vhdr
Extracting parameters from /Users/madhav/PSY197B/data/sj04/eeg/sj04_walk_attend.vhdr...
Setting channel info structure...
Reading 0 ... 1085959  =      0.000 ...  2171.918 secs...


/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: No coordinate information found for channels ['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(eeg_file, preload=True)
/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (131.00 Hz) will be stored.
  raw = mne.io.read_raw_brainvision(eeg_file, preload=True)
/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: Not setting positions of 10 misc channels found in montage:
['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z', 'x_dir', 'y_dir', 'z_dir']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating

    Downsampling from 500.0 Hz to 250 Hz
    Re-referencing to average of mastoids (TP9, TP10)
Applying a custom ('EEG',) reference.
    Removing channels: ['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z', 'x_dir', 'y_dir', 'z_dir']
    Filtering: 0.1–30 Hz
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hamming window
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 16501 samples (66.004 s)

    Marking bad channels (z>3.5): ['CP1']
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 30 sensor positio

/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: No coordinate information found for channels ['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(eeg_file, preload=True)
/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (131.00 Hz) will be stored.
  raw = mne.io.read_raw_brainvision(eeg_file, preload=True)
/var/folders/np/k3b6l2l16hn2bfn29bwtmlnr0000gn/T/ipykernel_35169/277376240.py:35: RuntimeWarning: Not setting positions of 10 misc channels found in montage:
['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z', 'x_dir', 'y_dir', 'z_dir']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating

    Downsampling from 500.0 Hz to 250 Hz
    Re-referencing to average of mastoids (TP9, TP10)
Applying a custom ('EEG',) reference.
    Removing channels: ['32', 'R_X', 'R_Y', 'R_Z', 'L_X', 'L_Y', 'L_Z', 'x_dir', 'y_dir', 'z_dir']
    Filtering: 0.1–30 Hz
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hamming window
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 16501 samples (66.004 s)

    Marking bad channels (z>3.5): ['CP1']
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 30 sensor positio

In [14]:
for sj_num in subjects:
    print(f'\nProcessing ET — Subject {sj_num:02d}')

    for cond in conditions:
        label = cond['eeg_label']
        print(f'  Condition: {label}')

        sj_dir = os.path.join(source_dir_et, et_folder_map[label])

        def load_et(fname):
            path = os.path.join(sj_dir, fname)
            return pd.read_csv(path) if os.path.exists(path) else None

        annotations = load_et('annotations.csv')
        gaze        = load_et('gaze_positions.csv')
        fixations   = load_et('fixations.csv')
        saccades    = load_et('saccades.csv')
        imu         = load_et('imu.csv')

        if annotations is None or gaze is None:
            print('    Missing annotations or gaze — skipping'); continue

        # Parse triggers (Event_N format, nanosecond timestamps)
        triggers = parse_annotations(annotations)
        print(f'    Found {len(triggers)} trial triggers (1–200)')

        # Convert gaze timestamps to seconds
        gaze['timestamp_s'] = gaze['timestamp [ns]'] / 1e9
        print(f'    Gaze: {len(gaze)} samples loaded (no confidence filter — Neon/Invisible export)')

        # Convert fixation/saccade timestamps to seconds (space in column name)
        if fixations is not None:
            fixations['start_s'] = fixations['start timestamp [ns]'] / 1e9
            fixations['end_s']   = fixations['end timestamp [ns]']   / 1e9
        if saccades is not None:
            saccades['start_s']  = saccades['start timestamp [ns]']  / 1e9
            saccades['end_s']    = saccades['end timestamp [ns]']     / 1e9

        # Motion flagging from IMU
        motion_windows = None
        if imu is not None and any('gyro' in c.lower() for c in imu.columns):
            gyro_cols = [c for c in imu.columns if 'gyro' in c.lower()]
            imu['motion_magnitude'] = imu[gyro_cols].abs().max(axis=1)
            thresh         = imu['motion_magnitude'].mean() + 2 * imu['motion_magnitude'].std()
            motion_windows = imu[imu['motion_magnitude'] > thresh]['timestamp [ns]'].values / 1e9
            print(f'    IMU: flagged {len(motion_windows)} high-motion samples')

        # Epoch ET around each trigger
        tmin, tmax    = -0.2, 1.0
        epoch_records = []

        for _, row in triggers.iterrows():
            t0     = row['start_timestamp']  # seconds
            t_code = int(row['trigger'])

            gaze_win = gaze[gaze['timestamp_s'].between(t0 + tmin, t0 + tmax)]

            fix_win = fixations[
                (fixations['start_s'] <= t0 + tmax) &
                (fixations['end_s']   >= t0 + tmin)
            ] if fixations is not None else pd.DataFrame()

            sacc_win = saccades[
                (saccades['start_s'] <= t0 + tmax) &
                (saccades['end_s']   >= t0 + tmin)
            ] if saccades is not None else pd.DataFrame()

            in_motion = bool(
                motion_windows is not None and
                np.any((motion_windows >= t0 + tmin) & (motion_windows <= t0 + tmax))
            )

            epoch_records.append({
                'trialIdx'             : t_code,
                'trigger_time'         : t0,
                # Gaze in degrees (azimuth/elevation) — preferred over pixels for outdoor walking
                # as they represent true gaze direction independent of head position
                'gaze_n_samples'       : len(gaze_win),
                'gaze_mean_az'         : gaze_win['azimuth [deg]'].mean()    if len(gaze_win) else np.nan,
                'gaze_mean_el'         : gaze_win['elevation [deg]'].mean()  if len(gaze_win) else np.nan,
                'gaze_std_az'          : gaze_win['azimuth [deg]'].std()     if len(gaze_win) else np.nan,
                'gaze_std_el'          : gaze_win['elevation [deg]'].std()   if len(gaze_win) else np.nan,
                # Pixel coords kept as secondary reference
                'gaze_mean_x_px'       : gaze_win['gaze x [px]'].mean()      if len(gaze_win) else np.nan,
                'gaze_mean_y_px'       : gaze_win['gaze y [px]'].mean()      if len(gaze_win) else np.nan,
                # Fixations
                'n_fixations'          : len(fix_win),
                'mean_fix_duration'    : fix_win['duration [ms]'].mean()     if len(fix_win) else np.nan,
                'total_fix_duration'   : fix_win['duration [ms]'].sum()      if len(fix_win) else 0.0,
                # Saccades
                'n_saccades'           : len(sacc_win),
                'mean_sacc_amplitude'  : sacc_win['amplitude [deg]'].mean()  if len(sacc_win) else np.nan,
                'mean_sacc_duration'   : sacc_win['duration [ms]'].mean()    if len(sacc_win) else np.nan,
                # Quality
                'motion_artifact'      : in_motion,
            })

        et_epochs = pd.DataFrame(epoch_records)
        print(f'    Created {len(et_epochs)} ET epochs')

        out_path = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_ET_Prepro1.csv')
        et_epochs.to_csv(out_path, index=False)
        print(f'    Saved: {out_path}')

print('\nET preprocessing complete!')


Processing ET — Subject 04
  Condition: walk_attend
    Non-trial event codes excluded (check meanings): <StringArray>
['Event_201', 'Event_222', 'Event_202', 'Event_203', 'Event_204', 'Event_205']
Length: 6, dtype: str
    Found 1000 trial triggers (1–200)
    Gaze: 199843 samples loaded (no confidence filter — Neon/Invisible export)
    IMU: flagged 4007 high-motion samples
    Created 1000 ET epochs
    Saved: /Users/madhav/PSY197B/EEG_Prepro_Test/sj04_walk_attend_ET_Prepro1.csv
  Condition: walk_unattend
    Missing annotations or gaze — skipping

ET preprocessing complete!


In [15]:
for sj_num in subjects:
    print(f'\nFusing EEG+ET — Subject {sj_num:02d}')

    for cond in conditions:
        label = cond['eeg_label']
        print(f'  Condition: {label}')

        eeg_path = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_EEG_Prepro1-epo.fif')
        et_path  = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_ET_Prepro1.csv')

        if not os.path.exists(eeg_path):
            print(f'    Missing EEG: {eeg_path}'); continue
        if not os.path.exists(et_path):
            print(f'    Missing ET:  {et_path}');  continue

        epochs         = mne.read_epochs(eeg_path, preload=False, verbose=False)
        et             = pd.read_csv(et_path)
        eeg_event_list = epochs.events[:, 2]
        print(f'    EEG: {len(eeg_event_list)} epochs  |  ET: {len(et)} epochs')

        et_aligned = align_to_eeg_events(et, eeg_event_list, idx_col='trialIdx')
        if et_aligned is None:
            print('    Skipping — alignment failed'); continue

        # Merge ET features into EEG metadata
        fused_meta = epochs.metadata.copy().reset_index(drop=True) if epochs.metadata is not None else pd.DataFrame()
        for col in [c for c in et_aligned.columns if c != 'trialIdx']:
            fused_meta[col] = et_aligned[col].values

        # Derived fusion features
        if {'gaze_std_az', 'gaze_std_el'}.issubset(fused_meta.columns):
            fused_meta['gaze_entropy'] = (
                fused_meta['gaze_std_az'].fillna(0) + fused_meta['gaze_std_el'].fillna(0)
            )

        if {'n_saccades', 'mean_fix_duration'}.issubset(fused_meta.columns):
            fused_meta['oculomotor_load'] = (
                fused_meta['n_saccades'] / fused_meta['mean_fix_duration'].replace(0, np.nan)
            )

        if 'motion_artifact' in fused_meta.columns:
            fused_meta['reject_trial'] = fused_meta['motion_artifact'].astype(bool)

        epochs.metadata = fused_meta

        out_epo  = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_EEG_ET_Fused-epo.fif')
        out_meta = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_fused_metadata.csv')
        epochs.save(out_epo, overwrite=True)
        fused_meta.to_csv(out_meta, index=False)
        print(f'    Saved: {out_epo}')
        print(f'    Saved: {out_meta}')

print('\nEEG + ET fusion complete!')


Fusing EEG+ET — Subject 04
  Condition: walk_attend
    EEG: 940 epochs  |  ET: 1000 epochs
    SYNC SUCCESS
Replacing existing metadata with 24 columns
Overwriting existing file.
Loading data for 1 events and 301 original time points ...
Overwriting existing file.
Overwriting existing file.
Loading data for 940 events and 301 original time points ...
    Saved: /Users/madhav/PSY197B/EEG_Prepro_Test/sj04_walk_attend_EEG_ET_Fused-epo.fif
    Saved: /Users/madhav/PSY197B/EEG_Prepro_Test/sj04_walk_attend_fused_metadata.csv
  Condition: walk_unattend
    Missing ET:  /Users/madhav/PSY197B/EEG_Prepro_Test/sj04_walk_unattend_ET_Prepro1.csv

EEG + ET fusion complete!


### Fixation-level long format
Expand trial-level ET epochs into fixation-level rows (long format).
Each fixation within a trial becomes its own row, tagged with `trialIdx`.
This is needed for feeding fixation-level time series into models later.

In [16]:
for sj_num in subjects:
    print(f'\nFixation long-format — Subject {sj_num:02d}')

    for cond in conditions:
        label = cond['eeg_label']
        print(f'  Condition: {label}')

        sj_dir = os.path.join(source_dir_et, et_folder_map[label])

        ann_path = os.path.join(sj_dir, 'annotations.csv')
        fix_path = os.path.join(sj_dir, 'fixations.csv')

        if not os.path.exists(ann_path) or not os.path.exists(fix_path):
            print('    Missing annotations or fixations — skipping')
            continue

        annotations = pd.read_csv(ann_path)
        fixations   = pd.read_csv(fix_path)

        triggers = parse_annotations(annotations)
        print(f'    {len(triggers)} triggers, {len(fixations)} fixations')

        fixations['start_s'] = fixations['start timestamp [ns]'] / 1e9
        fixations['end_s']   = fixations['end timestamp [ns]']   / 1e9
        fixations['mid_s']   = (fixations['start_s'] + fixations['end_s']) / 2.0

        tmin, tmax = -0.2, 1.0
        long_records = []

        for trial_num, (_, trig_row) in enumerate(triggers.iterrows(), start=1):
            t0     = trig_row['start_timestamp']
            t_code = int(trig_row['trigger'])

            fix_win = fixations[
                (fixations['start_s'] <= t0 + tmax) &
                (fixations['end_s']   >= t0 + tmin)
            ]

            for fix_idx, (_, fix_row) in enumerate(fix_win.iterrows()):
                long_records.append({
                    'trial_num':       trial_num,
                    'trialIdx':        t_code,
                    'trigger_time':    t0,
                    'fixation_order':  fix_idx + 1,
                    'fixation_id':     int(fix_row['fixation id']),
                    'fix_start_s':     fix_row['start_s'],
                    'fix_end_s':       fix_row['end_s'],
                    'fix_mid_s':       fix_row['mid_s'],
                    'fix_duration_ms': fix_row['duration [ms]'],
                    'fix_x_px':        fix_row['fixation x [px]'],
                    'fix_y_px':        fix_row['fixation y [px]'],
                    'fix_rel_time_s':  fix_row['mid_s'] - t0,
                })

        et_long = pd.DataFrame(long_records)
        print(f'    Expanded to {len(et_long)} fixation-level rows '
              f'({et_long["trial_num"].nunique()} unique trials)')

        out_path = os.path.join(dest_dir, f'sj{sj_num:02d}_{label}_ET_Long.csv')
        et_long.to_csv(out_path, index=False)
        print(f'    Saved: {out_path}')

print('\nFixation long-format export complete!')


Fixation long-format — Subject 04
  Condition: walk_attend
    Non-trial event codes excluded (check meanings): <StringArray>
['Event_201', 'Event_222', 'Event_202', 'Event_203', 'Event_204', 'Event_205']
Length: 6, dtype: str
    1000 triggers, 2766 fixations
    Expanded to 2569 fixation-level rows (923 unique trials)
    Saved: /Users/madhav/PSY197B/EEG_Prepro_Test/sj04_walk_attend_ET_Long.csv
  Condition: walk_unattend
    Missing annotations or fixations — skipping

Fixation long-format export complete!
